In [ ]:
from urllib.request import urlretrieve
from shutil import unpack_archive

adres = "https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip"
arsiv, _ = urlretrieve(adres)
unpack_archive(arsiv, "veri-seti", format="zip")

In [ ]:
import cv2

goruntu = cv2.imread("veri-seti/Fluo-N2DH-GOWT1/01/t000.tif", cv2.IMREAD_UNCHANGED)
print(goruntu.dtype)

uint8


In [ ]:
print(goruntu.min(), goruntu.max())

0 221


In [ ]:
import matplotlib.pyplot as plt

plt.hist(goruntu.ravel(), bins=range(257), log=True)
plt.xlabel("Piksel yoğunluğu")
plt.ylabel("Piksel sayısı (log ölçek)")
plt.title("01/t000.tif histogramı")
plt.show()

In [ ]:
plt.imshow(goruntu, cmap="gray", vmin=0, vmax=255)
plt.title("01/t000.tif")
plt.axis("off")
plt.show()

"01/t000.tif" görüntüsünde piksel yoğunlukları 0–221 arasında değişmektedir. Histogramda düşük yoğunlukların baskın olması, görüntüdeki koyu arka planla uyumludur. Bazı hücrelerin arka planla düşük kontrast göstermesi, bu hücrelerin ayırt edilmesini zorlaştırmaktadır.

In [ ]:
goruntu_02 = cv2.imread("veri-seti/Fluo-N2DH-GOWT1/02/t000.tif", cv2.IMREAD_UNCHANGED)
print("Veri tipi:", goruntu_02.dtype, "Minimum:", goruntu_02.min(), "Maksimum:", goruntu_02.max())

plt.hist(goruntu_02.ravel(), bins=range(257), log=True)
plt.xlabel("Piksel yoğunluğu")
plt.ylabel("Piksel sayısı (log ölçek)")
plt.title("02/t000.tif histogramı")
plt.show()

In [ ]:
plt.imshow(goruntu_02, cmap="gray", vmin=0, vmax=255)
plt.title("02/t000.tif")
plt.axis("off")
plt.show()

"02/t000.tif" görüntüsünde piksel yoğunlukları 0–155 arasında değişmektedir. Histogramda düşük yoğunluklar baskındır. Bazı hücreler koyu arka plandan güçlükle seçilirken sol üstte birbirine temas ediyor gibi görünen hücrelerin sınırlarını ayırt etmek de zordur.

In [ ]:
print("01/t000.tif", goruntu.shape)
print("02/t000.tif", goruntu_02.shape)

01/t000.tif (1024, 1024)
02/t000.tif (1024, 1024)


In [ ]:
ogrenci_no_son_5 = "09017"
seed = int(ogrenci_no_son_5) + 2026
print(seed)

11043


In [ ]:
import random

rng = random.Random(seed)

while True:
    dogrulama = sorted(rng.sample(range(92), 5))
    adaylar = [
        kare for kare in range(92)
        if all(abs(kare - d) >= 6 for d in dogrulama)
    ]
    if len(adaylar) >= 10:
        break

gelistirme = sorted(rng.sample(adaylar, 10))

print("Doğrulama:", dogrulama)
print("Geliştirme:", gelistirme)

Doğrulama: [5, 11, 15, 67, 68]
Geliştirme: [22, 27, 29, 34, 37, 48, 57, 78, 88, 89]


In [ ]:
test_rng = random.Random(seed)

while True:
    a, b, c = sorted(test_rng.sample(range(88), 3))
    if b - a >= 10 and c - b >= 10:
        break

test_bloklari = [list(range(i, i + 5)) for i in (a, b, c)]
print(test_bloklari)

[[35, 36, 37, 38, 39], [56, 57, 58, 59, 60], [77, 78, 79, 80, 81]]


In [ ]:
import pandas as pd

satirlar = []
for kareler, grup, seri in [
    (gelistirme, "gelistirme", "01"),
    (dogrulama, "dogrulama", "01"),
    (sum(test_bloklari, []), "test", "02"),
]:
    for kare in kareler:
        satirlar.append([grup, seri, kare, f"t{kare:03d}.tif", seed])

secim = pd.DataFrame(satirlar, columns=["grup", "seri", "kare", "dosya", "seed"])
secim.to_csv("selected_frames.csv", index=False)
print(secim["grup"].value_counts())

grup
test          15
gelistirme    10
dogrulama      5
Name: count, dtype: int64


In [ ]:
import numpy as np

referanslar = secim[secim["grup"].isin(["gelistirme", "dogrulama"])].copy()
sayilar = []

for kare in referanslar["kare"]:
    yol = f"veri-seti/Fluo-N2DH-GOWT1/01_GT/TRA/man_track{kare:03d}.tif"
    etiket = cv2.imread(yol, cv2.IMREAD_UNCHANGED)
    if etiket is None:
        raise FileNotFoundError(yol)
    sayilar.append(len(np.unique(etiket[etiket > 0])))

referanslar["referans_sayi"] = sayilar
display(referanslar[["grup", "dosya", "referans_sayi"]])

,grup,dosya,referans_sayi
0,gelistirme,t022.tif,24
1,gelistirme,t027.tif,24
2,gelistirme,t029.tif,24
3,gelistirme,t034.tif,24
4,gelistirme,t037.tif,24
5,gelistirme,t048.tif,24
6,gelistirme,t057.tif,21
7,gelistirme,t078.tif,20
8,gelistirme,t088.tif,21
9,gelistirme,t089.tif,21


In [ ]:
goruntu_22 = cv2.imread("veri-seti/Fluo-N2DH-GOWT1/01/t022.tif", cv2.IMREAD_UNCHANGED)
esik, maske = cv2.threshold(goruntu_22, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

plt.imshow(maske, cmap="gray")
plt.title(f"t022.tif – Otsu maskesi (eşik = {esik:.0f})")
plt.axis("off")
plt.show()

In [ ]:
etiket_sayisi, bilesenler = cv2.connectedComponents(maske, connectivity=8)
print("Baseline tahmini:", etiket_sayisi - 1)

Baseline tahmini: 932


In [ ]:
from skimage.color import label2rgb

plt.imshow(label2rgb(bilesenler, bg_label=0, bg_color="black"))
plt.title(f"t022.tif – Bağlı bölgeler ({etiket_sayisi - 1})")
plt.axis("off")
plt.show()

"t022.tif" görüntüsünde Otsu eşik değeri 31 olarak bulundu. Referans hücre sayısı 24 iken başlangıç yöntemi 932 bağlı bölge saydı. Bazı hücrelerin maskede parçalanması, bu görüntüdeki 908 fazla sayımın nedenlerinden biridir.

In [ ]:
def baseline_say(goruntu):
    _, maske = cv2.threshold(goruntu, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    etiket_sayisi, _ = cv2.connectedComponents(maske, connectivity=8)
    return etiket_sayisi - 1

baseline = referanslar.loc[
    referanslar["grup"] == "gelistirme", ["dosya", "referans_sayi"]
].copy()

baseline["tahmin"] = [
    baseline_say(cv2.imread(f"veri-seti/Fluo-N2DH-GOWT1/01/{dosya}", cv2.IMREAD_UNCHANGED))
    for dosya in baseline["dosya"]
]
baseline["mutlak_hata"] = (baseline["tahmin"] - baseline["referans_sayi"]).abs()
display(baseline)
print("Baseline MAE:", baseline["mutlak_hata"].mean())

,dosya,referans_sayi,tahmin,mutlak_hata
0,t022.tif,24,932,908
1,t027.tif,24,953,929
2,t029.tif,24,914,890
3,t034.tif,24,886,862
4,t037.tif,24,952,928
5,t048.tif,24,872,848
6,t057.tif,21,733,712
7,t078.tif,20,692,672
8,t088.tif,21,745,724
9,t089.tif,21,725,704


Baseline MAE: 817.7


In [ ]:
deneyler = pd.DataFrame([{
    "deney": "E01",
    "yogunluk": "Orijinal",
    "filtre": "Yok",
    "segmentasyon": "Otsu",
    "morfoloji": "Yok",
    "parametreler": "8-komşuluk, alan filtresi yok",
    "gelistirme_MAE": baseline["mutlak_hata"].mean()
}])
deneyler.to_csv("experiments.csv", index=False)
display(deneyler)

,deney,yogunluk,filtre,segmentasyon,morfoloji,parametreler,gelistirme_MAE
0,E01,Orijinal,Yok,Otsu,Yok,"8-komşuluk, alan filtresi yok",817.7


In [ ]:
esitlenmis = cv2.equalizeHist(goruntu_22)

"t022.tif" görüntüsünde histogram eşitleme sonrası sönük hücreler daha görünür hâle geldi. Arka plandaki grenli doku ve parlaklık farklılıkları da belirginleşti. Histogramda sıfırdaki tepe korunurken bazı yoğunluk değerleri daha parlak seviyelere taşındı.

In [ ]:
def gelistirmede_olc(donusum):
    sonuc = baseline[["dosya", "referans_sayi"]].copy()
    tahminler = []

    for dosya in sonuc["dosya"]:
        yol = f"veri-seti/Fluo-N2DH-GOWT1/01/{dosya}"
        goruntu = cv2.imread(yol, cv2.IMREAD_UNCHANGED)
        tahminler.append(baseline_say(donusum(goruntu)))

    sonuc["tahmin"] = tahminler
    sonuc["mutlak_hata"] = (sonuc["tahmin"] - sonuc["referans_sayi"]).abs()
    return sonuc

e02 = gelistirmede_olc(cv2.equalizeHist)
print("E02 MAE:", e02["mutlak_hata"].mean())

E02 MAE: 24208.8


In [ ]:
e02_kaydi = deneyler.loc[deneyler["deney"] == "E01"].copy()
e02_kaydi["deney"] = "E02"
e02_kaydi["yogunluk"] = "Histogram eşitleme"
e02_kaydi["gelistirme_MAE"] = e02["mutlak_hata"].mean()

deneyler = pd.concat([deneyler, e02_kaydi], ignore_index=True)
deneyler = deneyler.drop_duplicates("deney", keep="last")
deneyler.to_csv("experiments.csv", index=False)
display(deneyler[["deney", "yogunluk", "gelistirme_MAE"]])

,deney,yogunluk,gelistirme_MAE
0,E01,Orijinal,817.7
1,E02,Histogram eşitleme,24208.8


In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_22 = clahe.apply(goruntu_22)

ornekler = [
    ("Orijinal", goruntu_22),
    ("Histogram eşitleme", esitlenmis),
    ("CLAHE (2,0; 8×8 bölge)", clahe_22),
]

fig, eksen = plt.subplots(2, 3, figsize=(12, 6), sharey="row")
for sutun, (baslik, ornek) in enumerate(ornekler):
    eksen[0, sutun].imshow(ornek, cmap="gray", vmin=0, vmax=255)
    eksen[0, sutun].set_title(baslik)
    eksen[0, sutun].axis("off")
    eksen[1, sutun].hist(ornek.ravel(), bins=range(257), log=True)
    eksen[1, sutun].set_xlabel("Piksel yoğunluğu")

eksen[1, 0].set_ylabel("Piksel sayısı (log ölçek)")
fig.suptitle("t022.tif – Yoğunluk dönüşümlerinin karşılaştırılması")
plt.tight_layout()
plt.show()

"t022.tif" görüntüsünde piksellerin çoğu düşük yoğunluklarda toplanmıştır. Global histogram eşitleme arka plandaki benekleri de belirginleştirdiği için, CLAHE’de görüntüyü 8×8 bölgeye ayırıp kontrast sınırını 2,0 seçerek yerel kontrast artışını sınırlamayı denedik.

In [ ]:
e03 = gelistirmede_olc(clahe.apply)

e03_kaydi = deneyler.loc[deneyler["deney"] == "E02"].copy()
e03_kaydi["deney"] = "E03"
e03_kaydi["yogunluk"] = "CLAHE (clipLimit=2.0, 8×8)"
e03_kaydi["gelistirme_MAE"] = e03["mutlak_hata"].mean()

deneyler = pd.concat([deneyler, e03_kaydi], ignore_index=True)
deneyler = deneyler.drop_duplicates("deney", keep="last")
deneyler.to_csv("experiments.csv", index=False)

display(deneyler[["deney", "yogunluk", "gelistirme_MAE"]])

,deney,yogunluk,gelistirme_MAE
0,E01,Orijinal,817.7
1,E02,Histogram eşitleme,24208.8
2,E03,"CLAHE (clipLimit=2.0, 8×8)",617.4


Aynı 10 geliştirme görüntüsünde en düşük ortalama mutlak sayım hatasını CLAHE verdi (617,4). Orijinal görüntülerde hata 817,7, global histogram eşitlemede ise 24.208,8 oldu. CLAHE, orijinal görüntülere göre hatayı yaklaşık %24,5 azalttı.

In [ ]:
def gaussian_3(goruntu):
    return cv2.GaussianBlur(clahe.apply(goruntu), (3, 3), 0)

e04 = gelistirmede_olc(gaussian_3)
print("E04 MAE (Gaussian 3×3):", e04["mutlak_hata"].mean())

E04 MAE (Gaussian 3×3): 78.3


In [ ]:
def gaussian_5(goruntu):
    return cv2.GaussianBlur(clahe.apply(goruntu), (5, 5), 0)

e05 = gelistirmede_olc(gaussian_5)
print("E05 MAE (Gaussian 5×5):", e05["mutlak_hata"].mean())

E05 MAE (Gaussian 5×5): 31.5


In [ ]:
def medyanli(goruntu):
    return cv2.medianBlur(clahe.apply(goruntu), 5)

e06 = gelistirmede_olc(medyanli)
print("E06 MAE (Medyan 5×5):", e06["mutlak_hata"].mean())

E06 MAE (Medyan 5×5): 30.9


In [ ]:
cekirdek = np.array([[0, -1, 0],
                     [-1, 5, -1],
                     [0, -1, 0]], dtype=np.float32)

def keskinlestir(goruntu):
    return cv2.filter2D(clahe.apply(goruntu), -1, cekirdek)

e07 = gelistirmede_olc(keskinlestir)
print("E07 MAE (Keskinleştirme 3×3):", e07["mutlak_hata"].mean())

E07 MAE (Keskinleştirme 3×3): 19694.2


In [ ]:
filtre_deneyleri = [
    ("E04", "Gaussian 3×3 (sigma=0)", e04),
    ("E05", "Gaussian 5×5 (sigma=0)", e05),
    ("E06", "Medyan 5×5", e06),
    ("E07", "Keskinleştirme 3×3 [[0,-1,0],[-1,5,-1],[0,-1,0]]", e07),
]

kayitlar = []
for deney, filtre, sonuc in filtre_deneyleri:
    kayit = deneyler.loc[deneyler["deney"] == "E03"].copy()
    kayit["deney"] = deney
    kayit["filtre"] = filtre
    kayit["gelistirme_MAE"] = sonuc["mutlak_hata"].mean()
    kayitlar.append(kayit)

deneyler = pd.concat([deneyler, *kayitlar], ignore_index=True)
deneyler = deneyler.drop_duplicates("deney", keep="last")
deneyler.to_csv("experiments.csv", index=False)
display(deneyler[["deney", "yogunluk", "filtre", "gelistirme_MAE"]])

,deney,yogunluk,filtre,gelistirme_MAE
0,E01,Orijinal,Yok,817.7
1,E02,Histogram eşitleme,Yok,24208.8
2,E03,"CLAHE (clipLimit=2.0, 8×8)",Yok,617.4
3,E04,"CLAHE (clipLimit=2.0, 8×8)",Gaussian 3×3 (sigma=0),78.3
4,E05,"CLAHE (clipLimit=2.0, 8×8)",Gaussian 5×5 (sigma=0),31.5
5,E06,"CLAHE (clipLimit=2.0, 8×8)",Medyan 5×5,30.9
6,E07,"CLAHE (clipLimit=2.0, 8×8)","Keskinleştirme 3×3 [[0,-1,0],[-1,5,-1],[0,-1,0]]",19694.2


In [ ]:
filtre_ornekleri = [
    ("Filtresiz", clahe_22),
    ("Gaussian 3×3", cv2.GaussianBlur(clahe_22, (3, 3), 0)),
    ("Gaussian 5×5", cv2.GaussianBlur(clahe_22, (5, 5), 0)),
    ("Medyan 5×5", cv2.medianBlur(clahe_22, 5)),
    ("Keskinleştirme 3×3", cv2.filter2D(clahe_22, -1, cekirdek)),
]

fig, eksen = plt.subplots(1, 5, figsize=(15, 4))
for ax, (baslik, goruntu) in zip(eksen, filtre_ornekleri):
    ax.imshow(goruntu, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    ax.set_title(baslik)
    ax.axis("off")

fig.suptitle("t022.tif – CLAHE sonrası filtre karşılaştırması")
plt.tight_layout()
plt.show()

"t022.tif" görüntüsünde Gaussian ve medyan filtreler ince benekleri bir miktar azalttı. İki 5×5 filtrenin hücre sınırlarına etkisi görsel olarak benzerdi ve 10 görüntüdeki MAE değerleri de yakındı (Gaussian 31,5, medyan 30,9). En belirgin fark keskinleştirmede görüldü: arka plandaki benekler belirginleşti ve MAE, filtresiz duruma göre 617,4’ten 19.694,2’ye yükseldi.

In [ ]:
esikler = []
for dosya in e06["dosya"]:
    yol = f"veri-seti/Fluo-N2DH-GOWT1/01/{dosya}"
    goruntu = cv2.imread(yol, cv2.IMREAD_UNCHANGED)
    filtreli = cv2.medianBlur(clahe.apply(goruntu), 5)
    esik, _ = cv2.threshold(filtreli, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    esikler.append(esik)

sabit_esik = int(np.floor(np.median(esikler)))
print("E08 için sabit eşik:", sabit_esik)

E08 için sabit eşik: 36


In [ ]:
def sabit_maske(goruntu):
    filtreli = medyanli(goruntu)
    _, maske = cv2.threshold(filtreli, sabit_esik, 255, cv2.THRESH_BINARY)
    return maske

e08 = gelistirmede_olc(sabit_maske)
print(f"E08 MAE (sabit eşik {sabit_esik}):", e08["mutlak_hata"].mean())

E08 MAE (sabit eşik 36): 27.0


In [ ]:
def adaptif_maske(goruntu):
    filtreli = medyanli(goruntu)
    return cv2.adaptiveThreshold(
        filtreli, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, -2
    )

e09 = gelistirmede_olc(adaptif_maske)
print("E09 MAE (adaptif eşik):", e09["mutlak_hata"].mean())

E09 MAE (adaptif eşik): 8962.6


In [ ]:
kayitlar = []
for deney, yontem, sonuc in [
    ("E08", f"Sabit eşik (T={sabit_esik})", e08),
    ("E09", "Adaptif Gaussian (31×31, C=-2)", e09),
]:
    kayit = deneyler.loc[deneyler["deney"] == "E06"].copy()
    kayit["deney"] = deney
    kayit["segmentasyon"] = yontem
    kayit["gelistirme_MAE"] = sonuc["mutlak_hata"].mean()
    kayitlar.append(kayit)

deneyler = pd.concat([deneyler, *kayitlar], ignore_index=True)
deneyler = deneyler.drop_duplicates("deney", keep="last")
deneyler.to_csv("experiments.csv", index=False)
display(deneyler[["deney", "segmentasyon", "gelistirme_MAE"]].tail(2))

,deney,segmentasyon,gelistirme_MAE
7,E08,Sabit eşik (T=36),27.0
8,E09,"Adaptif Gaussian (31×31, C=-2)",8962.6


In [ ]:
filtreli_22 = cv2.medianBlur(clahe_22, 5)

_, sabit_22 = cv2.threshold(filtreli_22, sabit_esik, 255, cv2.THRESH_BINARY)
otsu_esigi, otsu_22 = cv2.threshold(
    filtreli_22, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)
adaptif_22 = cv2.adaptiveThreshold(
    filtreli_22, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, -2
)

maskeler = [
    (f"Sabit eşik (T={sabit_esik})", sabit_22),
    (f"Otsu (T={otsu_esigi:.0f})", otsu_22),
    ("Adaptif Gaussian (31×31, C=−2)", adaptif_22),
]

fig, eksen = plt.subplots(1, 3, figsize=(11, 4))
for ax, (baslik, maske) in zip(eksen, maskeler):
    ax.imshow(maske, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    ax.set_title(baslik)
    ax.axis("off")

fig.suptitle("t022.tif – Segmentasyon yöntemlerinin karşılaştırılması")
plt.tight_layout()
plt.show()

"t022.tif" görüntüsünde sabit eşik (36) ve Otsu (38) benzer maskeler üretti. Sabit eşik, bazı soluk hücrelerde daha fazla pikseli ön plana aldı. Adaptif eşikleme ise arka planda çok sayıda küçük beyaz bölge oluşturdu. Bu bölgelerin ayrı nesneler olarak sayılması, adaptif eşiklemede gözlenen yüksek sayım hatasına katkıda bulunmuş olabilir.

In [ ]:
eleman_3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

def morfoloji_olc(eleman, *islemler):
    def uygula(goruntu):
        maske = sabit_maske(goruntu)
        for islem in islemler:
            maske = cv2.morphologyEx(maske, islem, eleman)
        return maske

    return gelistirmede_olc(uygula)

e10 = morfoloji_olc(eleman_3, cv2.MORPH_ERODE)
print("E10 MAE (3×3 erozyon):", e10["mutlak_hata"].mean())

E10 MAE (3×3 erozyon): 14.8


In [ ]:
e11 = morfoloji_olc(eleman_3, cv2.MORPH_DILATE)
print("E11 MAE (3×3 genişletme):", e11["mutlak_hata"].mean())

E11 MAE (3×3 genişletme): 6.9


In [ ]:
e12 = morfoloji_olc(eleman_3, cv2.MORPH_OPEN)
print("E12 MAE (3×3 açma):", e12["mutlak_hata"].mean())

E12 MAE (3×3 açma): 5.9


In [ ]:
e13 = morfoloji_olc(eleman_3, cv2.MORPH_CLOSE)
print("E13 MAE (3×3 kapama):", e13["mutlak_hata"].mean())

E13 MAE (3×3 kapama): 14.2


In [ ]:
eleman_5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
e14 = morfoloji_olc(eleman_5, cv2.MORPH_OPEN)
print("E14 MAE (5×5 açma):", e14["mutlak_hata"].mean())

E14 MAE (5×5 açma): 4.7


In [ ]:
morfoloji_deneyleri = [
    ("E10", "Erozyon, elips 3×3", e10),
    ("E11", "Genişletme, elips 3×3", e11),
    ("E12", "Açma, elips 3×3", e12),
    ("E13", "Kapama, elips 3×3", e13),
    ("E14", "Açma, elips 5×5", e14),
]

kayitlar = []
for deney, islem, sonuc in morfoloji_deneyleri:
    kayit = deneyler.loc[deneyler["deney"] == "E08"].copy()
    kayit["deney"] = deney
    kayit["morfoloji"] = islem
    kayit["gelistirme_MAE"] = sonuc["mutlak_hata"].mean()
    kayitlar.append(kayit)

deneyler = pd.concat([deneyler, *kayitlar], ignore_index=True)
deneyler = deneyler.drop_duplicates("deney", keep="last")
deneyler.to_csv("experiments.csv", index=False)
display(deneyler[["deney", "morfoloji", "gelistirme_MAE"]].tail(5))

,deney,morfoloji,gelistirme_MAE
9,E10,"Erozyon, elips 3×3",14.8
10,E11,"Genişletme, elips 3×3",6.9
11,E12,"Açma, elips 3×3",5.9
12,E13,"Kapama, elips 3×3",14.2
13,E14,"Açma, elips 5×5",4.7


In [ ]:
morfoloji_ornekleri = [
    ("E08 · İşlem öncesi", sabit_22),
    ("E10 · Erozyon 3×3", cv2.erode(sabit_22, eleman_3)),
    ("E11 · Genişletme 3×3", cv2.dilate(sabit_22, eleman_3)),
    ("E12 · Açma 3×3", cv2.morphologyEx(sabit_22, cv2.MORPH_OPEN, eleman_3)),
    ("E13 · Kapama 3×3", cv2.morphologyEx(sabit_22, cv2.MORPH_CLOSE, eleman_3)),
    ("E14 · Açma 5×5", cv2.morphologyEx(sabit_22, cv2.MORPH_OPEN, eleman_5)),
]

fig, eksen = plt.subplots(2, 3, figsize=(10, 7))
for ax, (baslik, maske) in zip(eksen.flat, morfoloji_ornekleri):
    ax.imshow(maske, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    ax.set_title(baslik)
    ax.axis("off")

fig.suptitle("t022.tif – Morfolojik işlemlerin sabit eşik maskesine etkisi")
plt.tight_layout()
plt.show()

"t022.tif" görüntüsünde büyük beyaz bölgeler benzer kaldı. Erozyon ince yapıları küçültürken genişletme bu yapıları kalınlaştırdı. Sağ alt köşedeki küçük beyaz iz, 5×5 açmada 3×3 açmaya göre daha belirgin biçimde azaldı. On geliştirme görüntüsünde açma için MAE, eleman boyutu 3×3’ten 5×5’e çıkarıldığında 5,9’dan 4,7’ye düştü.

In [ ]:
e15 = morfoloji_olc(eleman_5, cv2.MORPH_OPEN, cv2.MORPH_CLOSE)
e16 = morfoloji_olc(eleman_5, cv2.MORPH_CLOSE, cv2.MORPH_OPEN)

print("E15 MAE:", e15["mutlak_hata"].mean())
print("E16 MAE:", e16["mutlak_hata"].mean())

E15 MAE: 3.6
E16 MAE: 3.2


In [ ]:
kayitlar = []
for deney, islem, sonuc in [
    ("E15", "Açma → Kapama, elips 5×5", e15),
    ("E16", "Kapama → Açma, elips 5×5", e16),
]:
    kayit = deneyler.loc[deneyler["deney"] == "E08"].copy()
    kayit["deney"] = deney
    kayit["morfoloji"] = islem
    kayit["gelistirme_MAE"] = sonuc["mutlak_hata"].mean()
    kayitlar.append(kayit)

deneyler = pd.concat([deneyler, *kayitlar], ignore_index=True)
deneyler = deneyler.drop_duplicates("deney", keep="last")
deneyler.to_csv("experiments.csv", index=False)
display(deneyler[["deney", "morfoloji", "gelistirme_MAE"]].tail(2))

,deney,morfoloji,gelistirme_MAE
14,E15,"Açma → Kapama, elips 5×5",3.6
15,E16,"Kapama → Açma, elips 5×5",3.2


In [ ]:
def e16_maskesi(goruntu):
    kapali = cv2.morphologyEx(sabit_maske(goruntu), cv2.MORPH_CLOSE, eleman_5)
    return cv2.morphologyEx(kapali, cv2.MORPH_OPEN, eleman_5)

kok = "veri-seti/Fluo-N2DH-GOWT1"
birlesme_adaylari = []

for dosya in e16["dosya"]:
    goruntu = cv2.imread(f"{kok}/01/{dosya}", cv2.IMREAD_UNCHANGED)
    referans = cv2.imread(f"{kok}/01_GT/TRA/man_track{dosya[1:]}", cv2.IMREAD_UNCHANGED)
    _, bolgeler = cv2.connectedComponents(e16_maskesi(goruntu), connectivity=8)

    ortak = (bolgeler > 0) & (referans > 0)
    for bolge in np.unique(bolgeler[ortak]):
        kimlikler = np.unique(referans[ortak & (bolgeler == bolge)])
        if len(kimlikler) >= 2:
            birlesme_adaylari.append((dosya, int(bolge), kimlikler.tolist()))

print("Olası birleşme sayısı:", len(birlesme_adaylari))
print("İlk örnek:", next(iter(birlesme_adaylari), "Bulunmadı"))

Olası birleşme sayısı: 0
İlk örnek: Bulunmadı


E16 maskeleri, 10 geliştirme görüntüsünde referans hücre etiketleriyle karşılaştırıldı. Birden fazla referans hücresiyle örtüşen bağlı bölge bulunmadığı için bu aşamada ek bir hücre ayırma işlemi uygulanmadı.

In [ ]:
deneyler = pd.read_csv("experiments.csv")
display(deneyler)

adaylar = deneyler[deneyler["deney"].isin(["E14", "E15", "E16"])]
adaylar = adaylar.sort_values("gelistirme_MAE")
display(adaylar)

,deney,yogunluk,filtre,segmentasyon,morfoloji,parametreler,gelistirme_MAE
0,E01,Orijinal,Yok,Otsu,Yok,"8-komşuluk, alan filtresi yok",817.7
1,E02,Histogram eşitleme,Yok,Otsu,Yok,"8-komşuluk, alan filtresi yok",24208.8
2,E03,"CLAHE (clipLimit=2.0, 8×8)",Yok,Otsu,Yok,"8-komşuluk, alan filtresi yok",617.4
3,E04,"CLAHE (clipLimit=2.0, 8×8)",Gaussian 3×3 (sigma=0),Otsu,Yok,"8-komşuluk, alan filtresi yok",78.3
4,E05,"CLAHE (clipLimit=2.0, 8×8)",Gaussian 5×5 (sigma=0),Otsu,Yok,"8-komşuluk, alan filtresi yok",31.5
5,E06,"CLAHE (clipLimit=2.0, 8×8)",Medyan 5×5,Otsu,Yok,"8-komşuluk, alan filtresi yok",30.9
6,E07,"CLAHE (clipLimit=2.0, 8×8)","Keskinleştirme 3×3 [[0,-1,0],[-1,5,-1],[0,-1,0]]",Otsu,Yok,"8-komşuluk, alan filtresi yok",19694.2
7,E08,"CLAHE (clipLimit=2.0, 8×8)",Medyan 5×5,Sabit eşik (T=36),Yok,"8-komşuluk, alan filtresi yok",27.0
8,E09,"CLAHE (clipLimit=2.0, 8×8)",Medyan 5×5,"Adaptif Gaussian (31×31, C=-2)",Yok,"8-komşuluk, alan filtresi yok",8962.6
9,E10,"CLAHE (clipLimit=2.0, 8×8)",Medyan 5×5,Sabit eşik (T=36),"Erozyon, elips 3×3","8-komşuluk, alan filtresi yok",14.8


,deney,yogunluk,filtre,segmentasyon,morfoloji,parametreler,gelistirme_MAE
15,E16,"CLAHE (clipLimit=2.0, 8×8)",Medyan 5×5,Sabit eşik (T=36),"Kapama → Açma, elips 5×5","8-komşuluk, alan filtresi yok",3.2
14,E15,"CLAHE (clipLimit=2.0, 8×8)",Medyan 5×5,Sabit eşik (T=36),"Açma → Kapama, elips 5×5","8-komşuluk, alan filtresi yok",3.6
13,E14,"CLAHE (clipLimit=2.0, 8×8)",Medyan 5×5,Sabit eşik (T=36),"Açma, elips 5×5","8-komşuluk, alan filtresi yok",4.7


In [ ]:
fig, eksen = plt.subplots(2, 5, figsize=(15, 8), constrained_layout=True)

for ax, satir in zip(eksen.flat, e16.itertuples(index=False)):
    goruntu = cv2.imread(f"{kok}/01/{satir.dosya}", cv2.IMREAD_UNCHANGED)
    ax.imshow(goruntu, cmap="gray", vmin=0, vmax=255)
    ax.contour(e16_maskesi(goruntu), levels=[127], colors="orange", linewidths=0.7)
    ax.set_title(f"{satir.dosya}\nReferans {satir.referans_sayi} · Tahmin {satir.tahmin}")
    ax.axis("off")

fig.suptitle("E16 – Geliştirme görüntülerinde segmentasyon sınırları")
plt.show()

E16, 10 geliştirme görüntüsünün 7’sinde referanstan az, 3’ünde fazla hücre saydı (MAE 3,2). En büyük fark t048.tif görüntüsünde görüldü: referans sayı 24 iken tahmin 17 oldu. Bu görüntüde parlak yapıların çoğu turuncu sınırlarla çevrelenirken alt yarıdaki bazı soluk yapılar maskenin dışında kaldı. Bu durum eksik sayıma katkıda bulunmuş olabilir.

In [ ]:
islem_siralari = {
    "E14": [cv2.MORPH_OPEN],
    "E15": [cv2.MORPH_OPEN, cv2.MORPH_CLOSE],
    "E16": [cv2.MORPH_CLOSE, cv2.MORPH_OPEN],
}

def aday_say(goruntu, deney):
    maske = sabit_maske(goruntu)
    for islem in islem_siralari[deney]:
        maske = cv2.morphologyEx(maske, islem, eleman_5)
    return baseline_say(maske)

for deney in islem_siralari:
    print(deney, aday_say(goruntu_22, deney))

E14 27
E15 19
E16 20


In [ ]:
secili_kareler = pd.read_csv("selected_frames.csv")
dogrulama = secili_kareler.query("grup == 'dogrulama' and seri == 1")
print(dogrulama["dosya"].tolist())

['t005.tif', 't011.tif', 't015.tif', 't067.tif', 't068.tif']


In [ ]:
from time import perf_counter

referans_sayilari = (
    referanslar.query("grup == 'dogrulama'")
    .set_index("dosya")["referans_sayi"]
)
kayitlar = []

for dosya in dogrulama["dosya"]:
    goruntu = cv2.imread(f"{kok}/01/{dosya}", cv2.IMREAD_UNCHANGED)
    for deney in islem_siralari:
        baslangic = perf_counter()
        tahmin = aday_say(goruntu, deney)
        sure_ms = (perf_counter() - baslangic) * 1000
        kayitlar.append((deney, dosya, tahmin - referans_sayilari[dosya], sure_ms))

detay = pd.DataFrame(kayitlar, columns=["deney", "dosya", "hata", "sure_ms"])
sonuclar = detay.groupby("deney").agg(
    MAE=("hata", lambda h: h.abs().mean()),
    RMSE=("hata", lambda h: np.sqrt((h**2).mean())),
    ortalama_sure_ms=("sure_ms", "mean"),
).sort_values(["MAE", "RMSE", "ortalama_sure_ms"])

display(sonuclar.round(3))

,MAE,RMSE,ortalama_sure_ms
deney,,,
E16,3.6,3.847,16.493
E15,3.6,3.847,16.568
E14,8.6,10.421,14.301


In [ ]:
dogrulama_tablosu = sonuclar.reset_index()
dogrulama_tablosu["secim"] = np.where(
    dogrulama_tablosu["deney"] == "E15", "Seçildi", "Seçilmedi"
)

dogrulama_tablosu.to_csv("validation_results.csv", index=False)
display(dogrulama_tablosu.round(3))

,deney,MAE,RMSE,ortalama_sure_ms,secim
0,E16,3.6,3.847,16.493,Seçilmedi
1,E15,3.6,3.847,16.568,Seçildi
2,E14,8.6,10.421,14.301,Seçilmedi


Beş doğrulama görüntüsünde E15 ve E16 aynı MAE (3,6) ve RMSE (3,847) değerlerini verdi. Her iki yöntem de aynı sayıda morfolojik işlem içerdiğinden, ilk doğrulama ölçümünde görüntü başına ortalama işlem süresi daha kısa olan E15 nihai yöntem olarak seçildi. E14’ün doğrulama MAE’si 8,6 oldu.

In [ ]:
acma_22 = cv2.morphologyEx(sabit_22, cv2.MORPH_OPEN, eleman_5)
kapama_22 = cv2.morphologyEx(acma_22, cv2.MORPH_CLOSE, eleman_5)

adimlar = [
    ("Orijinal", goruntu_22),
    ("CLAHE (2,0 · 8×8 bölge)", clahe_22),
    ("Medyan filtre (5×5)", filtreli_22),
    ("Sabit eşik (T=36)", sabit_22),
    ("Açma (5×5 elips)", acma_22),
    (f"Kapama (5×5 elips) · Sayım: {baseline_say(kapama_22)}", kapama_22),
]

fig, eksenler = plt.subplots(2, 3, figsize=(9, 6), constrained_layout=True)
for ax, (baslik, goruntu) in zip(eksenler.flat, adimlar):
    ax.imshow(goruntu, cmap="gray", vmin=0, vmax=255)
    ax.set_title(baslik)
    ax.axis("off")

fig.suptitle("t022.tif – E15 yönteminin işlem adımları")
plt.show()

"t022.tif" görüntüsünde sabit eşik (T=36) uygulandığında bazı soluk yapılar maskenin dışında kaldı. Açma işleminden sonra sağ alttaki küçük beyaz iz küçüldü. Kapama sonrasında E15, 19 bağlı bölge saydı. Bu sayı, 24 hücrelik referansın 5 altındadır.

In [ ]:
import json

final_pipeline = {
    "deney": "E15",
    "islem_sirasi": ["CLAHE", "medyan", "sabit_esik", "acma", "kapama", "sayim"],
    "clahe": {
        "clipLimit": clahe.getClipLimit(),
        "tileGridSize": list(clahe.getTilesGridSize())
    },
    "medyan_boyutu": 5,
    "sabit_esik": int(sabit_esik),
    "esik_turu": "THRESH_BINARY",
    "morfoloji": {
        "eleman": "elips",
        "boyut": [5, 5],
        "her_islem_icin_tekrar": 1
    },
    "sayim_fonksiyonu": "baseline_say",
    "sayim": {"komsuluk": 8, "alan_filtresi": False},
    "baseline": {"esik": "Otsu", "komsuluk": 8, "alan_filtresi": False}
}

with open("final_pipeline.json", "w", encoding="utf-8") as dosya:
    json.dump(final_pipeline, dosya, ensure_ascii=False, indent=2)

print("Kaydedildi: final_pipeline.json")

Kaydedildi: final_pipeline.json


In [ ]:
from pathlib import Path

kilit_yolu = Path("test_predictions_locked.csv")
test = secili_kareler.query("grup == 'test' and seri == 2").sort_values("kare")
assert len(test) == test["dosya"].nunique() == 15

if kilit_yolu.exists():
    kilitli = pd.read_csv(kilit_yolu)
    assert kilitli["dosya"].tolist() == test["dosya"].tolist()
    assert kilitli[["baseline_tahmin", "e15_tahmin"]].notna().all().all()
    print("15 kilitli test tahmini dosyadan okundu.")
else:
    kayitlar = []

    for satir in test.itertuples(index=False):
        yol = f"{kok}/02/{satir.dosya}"
        goruntu = cv2.imread(yol, cv2.IMREAD_UNCHANGED)
        if goruntu is None:
            raise FileNotFoundError(yol)

        baslangic = perf_counter()
        baseline_tahmin = baseline_say(goruntu)
        baseline_ms = (perf_counter() - baslangic) * 1000

        baslangic = perf_counter()
        e15_tahmin = aday_say(goruntu, "E15")
        e15_ms = (perf_counter() - baslangic) * 1000

        kayitlar.append((satir.kare, satir.dosya, baseline_tahmin,
                         e15_tahmin, baseline_ms, e15_ms))

    kilitli = pd.DataFrame(kayitlar, columns=[
        "kare", "dosya", "baseline_tahmin", "e15_tahmin",
        "baseline_ms", "e15_ms"
    ])
    kilitli.to_csv(kilit_yolu, index=False, mode="x")
    print("15 test görüntüsünün tahminleri kaydedildi.")

15 kilitli test tahmini dosyadan okundu.


In [ ]:
test_sonuclari = pd.read_csv("test_predictions_locked.csv")
assert len(test_sonuclari) == test_sonuclari["kare"].nunique() == 15

referanslar_test = []
for kare in test_sonuclari["kare"]:
    yol = f"{kok}/02_GT/TRA/man_track{kare:03d}.tif"
    etiket = cv2.imread(yol, cv2.IMREAD_UNCHANGED)
    if etiket is None:
        raise FileNotFoundError(yol)
    referanslar_test.append(len(np.unique(etiket[etiket > 0])))

test_sonuclari["referans_sayi"] = referanslar_test
for yontem in ["baseline", "e15"]:
    hata = test_sonuclari[f"{yontem}_tahmin"] - test_sonuclari["referans_sayi"]
    test_sonuclari[f"{yontem}_mutlak_hata"] = hata.abs()

display(test_sonuclari[[
    "kare", "referans_sayi", "baseline_tahmin", "e15_tahmin",
    "baseline_mutlak_hata", "e15_mutlak_hata"
]])


test_sonuclari.to_csv("test_results.csv", index=False)

,kare,referans_sayi,baseline_tahmin,e15_tahmin,baseline_mutlak_hata,e15_mutlak_hata
0,35,26,1813,27,1787,1
1,36,27,1874,22,1847,5
2,37,27,1889,17,1862,10
3,38,28,2008,15,1980,13
4,39,28,2004,19,1976,9
5,56,31,2213,21,2182,10
6,57,31,2189,21,2158,10
7,58,31,2222,20,2191,11
8,59,31,2217,19,2186,12
9,60,31,2081,19,2050,12


In [ ]:
ozet = []
for yontem, ad in [("baseline", "Baseline"), ("e15", "Nihai pipeline (E15)")]:
    hata = test_sonuclari[f"{yontem}_tahmin"] - test_sonuclari["referans_sayi"]
    ozet.append({
        "Yöntem": ad,
        "MAE": hata.abs().mean(),
        "RMSE": np.sqrt((hata ** 2).mean()),
        "Bias": hata.mean(),
        "Tam doğru sayım (%)": 100 * (hata == 0).mean(),
        "Ortalama süre (ms)": test_sonuclari[f"{yontem}_ms"].mean()
    })

display(pd.DataFrame(ozet).round(3))

,Yöntem,MAE,RMSE,Bias,Tam doğru sayım (%),Ortalama süre (ms)
0,Baseline,2054.467,2058.833,2054.467,0.0,3.313
1,Nihai pipeline (E15),10.733,11.457,-10.600,0.0,16.511


15 test görüntüsünde E15, ortalama mutlak sayım hatasını 2054,47’den 10,73’e düşürdü. Baseline tüm karelerde fazla sayarken E15, 15 karenin 14’ünde eksik saydı (Bias = −10,60). Her iki yöntemin de tam doğru sayım oranı %0’dı. Görüntü başına ortalama işlem süresi baseline için 3,31 ms, E15 için 16,51 ms olarak ölçüldü.

In [ ]:
ornekler = test_sonuclari.set_index("kare").loc[[35, 78, 80]]
fig, eksenler = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)

for ax, satir in zip(eksenler, ornekler.itertuples()):
    yol = f"{kok}/02/{satir.dosya}"
    goruntu = cv2.imread(yol, cv2.IMREAD_UNCHANGED)
    if goruntu is None:
        raise FileNotFoundError(yol)

    maske = sabit_maske(goruntu)
    for islem in islem_siralari["E15"]:
        maske = cv2.morphologyEx(maske, islem, eleman_5)

    etiket_sayisi, _ = cv2.connectedComponents(maske, connectivity=8)
    assert etiket_sayisi - 1 == satir.e15_tahmin, satir.dosya

    ax.imshow(goruntu, cmap="gray", vmin=0, vmax=255)
    ax.contour(maske, levels=[127], colors="orange", linewidths=0.7)
    ax.set_title(
        f"{satir.dosya} · Mutlak hata: {satir.e15_mutlak_hata}\n"
        f"Referans: {satir.referans_sayi} · E15: {satir.e15_tahmin}"
    )
    ax.axis("off")

fig.suptitle("Test görüntülerinde E15 sayım örnekleri")
plt.show()

"t035.tif" görüntüsünde E15, 26 hücreye karşılık 27 hücre sayarak referans sayıya en yakın tahmini vermiştir. "t078.tif" ve "t080.tif" görüntülerinde ise sırasıyla 17 ve 15 hücre eksik saymıştır. Bu iki görüntüde bazı soluk yapıların turuncu sınırların dışında kalması eksik sayıma katkıda bulunmuş olabilir.

In [ ]:
fig, eksenler = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, baslangic in zip(eksenler, [35, 56, 77]):
    blok = test_sonuclari[test_sonuclari["kare"].between(baslangic, baslangic + 4)]
    kare = blok["kare"]

    ax.plot(kare, blok["referans_sayi"], "o-", label="Gerçek sayı")
    ax.plot(kare, blok["baseline_tahmin"], "s--", label="Baseline")
    ax.plot(kare, blok["e15_tahmin"], "^:", label="E15")
    ax.set_title(f"Kare {baslangic}–{baslangic + 4}")
    ax.set_xlabel("Kare numarası")
    ax.set_xticks(kare)

eksenler[0].set_ylabel("Hücre sayısı (log ölçek)")
eksenler[0].set_yscale("log")
eksenler[0].legend()
plt.tight_layout()
plt.show()

35–39. karelerde gerçek sayı 26’dan 28’e çıkarken E15 tahmini 27’den 15’e düşüp 19’a yükseldi. 56–60. karelerde gerçek sayı 31’de sabitken E15, 19–21 hücre saydı. 77–81 bloğunun son iki karesinde gerçek sayı 29’da kalırken E15 tahmini 14’ten 22’ye çıktı. Soluk hücrelerin bazı karelerde eşik altında kalması bu dalgalanmalara katkıda bulunmuş olabilir. Baseline ise üç blokta da 1800’ün üzerinde hücre saydı.

**Deneysel Sonuçların Tartışılması**

**Yoğunluk dönüşümleri:** On geliştirme görüntüsünde en düşük sayım hatasını CLAHE verdi (MAE 617,4). Orijinal görüntülerde MAE 817,7, global histogram eşitlemede ise 24.208,8 oldu. t022.tif görüntüsünde global eşitleme arka plandaki benekleri belirginleştirirken CLAHE arka planı koyu tuttu.

**Filtreleme ve çekirdek boyutu:** CLAHE sonrasında Gaussian filtrenin çekirdek boyutunu 3×3’ten 5×5’e çıkarmak MAE’yi 78,3’ten 31,5’e düşürdü. Medyan 5×5 benzer bir sonuç verdi (MAE 30,9). t022.tif görüntüsünde her iki 5×5 filtre de ince benekleri azaltırken keskinleştirme bunları belirginleştirdi. Keskinleştirmede MAE 19.694,2’ye yükseldi.

**Segmentasyon:** CLAHE ve 5×5 medyan filtre sonrasında, 10 geliştirme görüntüsünde en düşük MAE’yi sabit eşik verdi (T=36, MAE 27,0). Otsu yakın bir sonuç verdi (30,9). t022.tif görüntüsünde bu iki yöntemin maskeleri benzerdi. Adaptif eşikleme ise arka planda çok sayıda küçük bölgeyi ön plana çıkardı. Bu bölgelerin ayrı hücreler olarak sayılması, adaptif yöntemdeki yüksek MAE’ye (8.962,6) katkıda bulunmuş olabilir.

**Morfolojik işlemler:** Açmada elips biçimli yapılandırma elemanını 3×3’ten 5×5’e büyütmek, geliştirme MAE’sini 5,9’dan 4,7’ye düşürdü. t022.tif görüntüsünün sağ altındaki küçük beyaz iz de 5×5 açmada daha az belirgindi. Aynı 5×5 elemanla işlem sırası da sonucu etkiledi. Önce açma, sonra kapama uygulandığında MAE 3,6, önce kapama, sonra açma uygulandığında ise 3,2 oldu.

**Ölçülebilir iyileşmeler:** E15’e giden adımlar arasında geliştirme MAE’sindeki en büyük düşüş, CLAHE sonrasında 5×5 medyan filtre uygulandığında görüldü (617,4’ten 30,9’a). Otsu yerine sabit eşik kullanmak hatayı 27,0’a, ardından 5×5 elipsle açma ve kapama uygulamak ise 3,6’ya düşürdü.

**Doğrulama ve test performansı:** E15’in MAE’si beş doğrulama görüntüsünde 3,6 iken 15 bağımsız test görüntüsünde 10,73’e yükseldi. Bu nedenle E15, doğrulamadaki sayım doğruluğunu testte koruyamadı. Test görüntülerinin 14’ünde eksik sayım yaptı ve görüntü başına ortalama 10,6 hücre eksik saydı.

**Baseline ile karşılaştırma:** E15, 15 test görüntüsünün tamamında referans sayıya baseline’dan daha yakın tahmin verdi. MAE 2054,47’den 10,73’e düştü. Bu iyileşmenin karşılığında görüntü başına ortalama işlem süresi 3,31 ms’den 16,51 ms’ye çıktı.

**Hatalı sayımların nedenleri:** "t022.tif" görüntüsünde bazı hücreler başlangıç maskesinde birden fazla parçaya ayrıldı. Her parçanın ayrı sayılması, baseline’ın 24 hücreye karşılık 932 tahmin üretmesine katkıda bulundu. E15’in testteki eksik sayımlarının olası nedenleri ise soluk hücrelerin sabit eşiğin altında kalması ve küçük bölgelerin morfolojik açma sırasında kaybolmasıdır.

**Başka görüntülere uygulanabilirlik:** Farklı mikroskop veya çekim koşullarında hücre parlaklığı ve arka plan değişebilir. Bu durumda sabit eşik (T=36) soluk hücreleri kaçırabilir veya arka planı hücre olarak algılayabilir. Görüntü çözünürlüğü değiştiğinde 5×5 filtre ve yapılandırma elemanının hücre boyutuna göre etkisi de değişir. Bu nedenle yöntemin başarısı yeni veri setinde ayrıca doğrulanmalıdır.